# CNN dari Dasar vs Transfer Learning untuk Klasifikasi Citra
**Tugas Individu – Pembelajaran Mesin**

**Nama:** Ahmad Furqon Ramadhani  
**NIM:** 452024611066
**Program Studi:** Teknik Informatika — Universitas Darussalam Gontor  
**Email:** afurqonramadhani24@student.cs.unida.gontor.ac.id

| Aspek | Detail |
|---|---|
| Dataset CNN | CIFAR-10 (Airplane vs Automobile) |
| Dataset Transfer Learning | Cats vs Dogs |
| Pretrained Model | MobileNetV2 (ImageNet) |
| Strategi | Feature Extraction |



In [1]:
import os
import pickle
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPU tersedia: []


In [ ]:
CATS_DOGS_DIR = "/kaggle/input/datasets/aleemaparakatta/cats-and-dogs-mini-dataset"
CIFAR_DIR     = "/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py"

---
## Eksperimen 1 — Arsitektur CNN Mandiri (Dataset: CIFAR-10)

### Karakteristik Dataset
CIFAR-10 merupakan sebuah dataset standardisasi (*benchmark*) yang memuat total 60.000 citra berwarna (RGB) dengan resolusi spasial 32×32 piksel yang terbagi ke dalam 10 kelas objek. Pada eksperimen pertama ini, ruang lingkup pengujian direduksi secara spesifik menjadi tugas klasifikasi biner dengan mengekstraksi dua kategori utama:
- **Kelas 0** – *Airplane* (Pesawat Terbang)
- **Kelas 1** – *Automobile* (Mobil)

**Skema Partisi Data:** Data didistribusikan secara proporsional menggunakan rasio pendekatan ±70:15:15, yang menghasilkan sebaran berupa 700 sampel untuk fase pelatihan (*training*), 150 sampel untuk fase validasi (*validation*), dan 150 sampel untuk fase pengujian (*testing*).

### Tahapan Pemrosesan Awal (*Preprocessing*)
- **Normalisasi Nilai Piksel:** Melakukan transformasi nilai intensitas piksel ke dalam rentang skala [0, 1] melalui operasi pembagian absolut dengan konstanta 255.0 untuk stabilitas komputasi dan mempercepat konvergensi gradien.
- **Restrukturisasi Dimensi Label:** Mengubah bentuk matriks label (*reshape*) menjadi format kolom tunggal guna menyesuaikan dengan arsitektur keluaran fungsi *loss* pada pemodelan klasifikasi biner.
- **Determinasi Augmentasi Data:** Proses augmentasi citra tidak diimplementasikan pada tahapan ini karena distribusi data pada set pelatihan dinilai sudah representatif dan seimbang (*balanced dataset*).

In [ ]:
def load_cifar10_batch(file_path):
    with open(file_path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
        d_decoded = {k.decode('utf8'): v for k, v in d.items()}
        data   = d_decoded['data']
        labels = d_decoded['labels']
        data   = data.reshape(10000, 3, 32, 32).transpose(0, 2, 3, 1)
        return data, np.array(labels)


train_data, train_labels = [], []
for i in range(1, 6):
    bx, by = load_cifar10_batch(os.path.join(CIFAR_DIR, f'data_batch_{i}'))
    train_data.append(bx)
    train_labels.append(by)

X_train_full = np.concatenate(train_data)
y_train_full = np.concatenate(train_labels)
X_test_full, y_test_full = load_cifar10_batch(os.path.join(CIFAR_DIR, 'test_batch'))


train_mask = (y_train_full == 0) | (y_train_full == 1)
test_mask  = (y_test_full  == 0) | (y_test_full  == 1)

x_train_bin = X_train_full[train_mask]
y_train_bin = y_train_full[train_mask].reshape(-1, 1)
x_test_bin  = X_test_full[test_mask]
y_test_bin  = y_test_full[test_mask].reshape(-1, 1)

print(f"Total data setelah filter: {len(x_train_bin)} train + {len(x_test_bin)} test")


X_train_cnn = x_train_bin[:700]  / 255.0;  y_train_cnn = y_train_bin[:700]
X_val_cnn   = x_train_bin[700:850] / 255.0; y_val_cnn   = y_train_bin[700:850]
X_test_cnn  = x_test_bin[:150]  / 255.0;  y_test_cnn  = y_test_bin[:150]

print(f"Train: {X_train_cnn.shape}, Val: {X_val_cnn.shape}, Test: {X_test_cnn.shape}")


fig, axes = plt.subplots(2, 8, figsize=(16, 4))
class_names_cnn = ['Airplane', 'Automobile']
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_cnn[i])
    ax.set_title(class_names_cnn[int(y_train_cnn[i][0])], fontsize=8)
    ax.axis('off')
plt.suptitle('Sampel Dataset CIFAR-10 (Airplane vs Automobile)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

unique, counts = np.unique(y_train_cnn, return_counts=True)
print(f"\nDistribusi kelas training: {dict(zip(['Airplane','Automobile'], counts))}")

### Arsitektur CNN dari Dasar

**Alasan desain arsitektur:**
- **2 blok Conv+Pool:** CIFAR-10 berukuran 32×32 sehingga terlalu banyak layer akan menghabiskan spatial resolution. Dua blok cukup untuk mengekstrak fitur low-level (tepi, tekstur) dan mid-level (bentuk).
- **BatchNormalization setelah Conv pertama:** Menstabilkan distribusi aktivasi dan mempercepat konvergensi.
- **Dropout(0.25) setelah blok kedua & Dropout(0.5) sebelum output:** Mengurangi risiko overfitting pada dataset yang relatif kecil (700 sampel).
- **Dense(64) → Dense(1, sigmoid):** Binary classification cukup dengan satu neuron output + sigmoid.
- **Adam lr=0.001:** Default Adam terbukti stabil untuk sebagian besar tugas klasifikasi.

In [ ]:

cnn_model = models.Sequential([

    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),


    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
], name='CNN_from_Scratch')

cnn_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()
cnn_total_params = cnn_model.count_params()
print(f"\nTotal parameter CNN: {cnn_total_params:,}")

print("\nMemulai pelatihan CNN From Scratch...")
start_time   = time.time()
cnn_history  = cnn_model.fit(
    X_train_cnn, y_train_cnn,
    epochs=10,
    batch_size=32,
    validation_data=(X_val_cnn, y_val_cnn),
    verbose=1
)
cnn_time_total = time.time() - start_time
print(f"Selesai! Waktu training CNN Scratch: {cnn_time_total:.2f} detik")

---
## Eksperimen 2 — Implementasi Transfer Learning (Dataset: Cats vs Dogs)

### Rincian Dataset
Dataset *Cats vs Dogs* dipilih sebagai basis klasifikasi biner pada tahap ini, yang terdiri dari:
- **Kelas 0:** Kucing (*Cat*)
- **Kelas 1:** Anjing (*Dog*)

**Referensi Data:** [Kaggle - Cats and Dogs Mini Dataset](https://www.kaggle.com/datasets/aleemaparakatta/cats-and-dogs-mini-dataset)

> **Informasi Dataset Eksperimen 1:** Dataset CIFAR-10 sebelumnya merujuk pada karya Krizhevsky & Hinton (2009). Dalam eksperimen ini, versi dataset ditarik dari repositori [pankrzysiu/cifar10-python](https://www.kaggle.com/datasets/pankrzysiu/cifar10-python) di platform Kaggle.

**Proporsi Pembagian Data:** Data didistribusikan secara proporsional menjadi 70% *training*, 15% *validation*, dan 15% *testing*, menggunakan fungsi `image_dataset_from_directory` untuk efisiensi manajemen *batch*.

### Pemilihan Model: MobileNetV2
**Rasionalisasi Penggunaan MobileNetV2:**
1. **Komputasi Ringan:** Memiliki sekitar 3,4 juta parameter, menjadikannya arsitektur yang efisien tanpa mengorbankan performa *baseline* dari *ImageNet*.
2. **Ramah Spesifikasi Perangkat:** Sangat ideal untuk dieksekusi pada lingkungan perangkat keras dengan keterbatasan memori (CPU/GPU standar).
3. **Kecocokan Domain Ekstraksi:** Karena arsitektur asal dilatih menggunakan *ImageNet* yang memuat banyak kategori hewan, representasi fiturnya sangat relevan untuk mengenali struktur anatomi kucing dan anjing.
4. **Kemudahan Integrasi:** Menawarkan kompatibilitas yang tinggi dengan ekosistem API Keras.

**Pendekatan Pelatihan: Ekstraksi Fitur (*Feature Extraction*)**
Bobot pada lapisan dasar MobileNetV2 sepenuhnya dikunci (*freeze* melalui parameter `trainable=False`). Siklus *training* hanya difokuskan untuk mengoptimalkan lapisan pengklasifikasi (*classifier*) baru di bagian akhir. Metode konservatif ini diterapkan untuk meminimalisasi *overfitting*, mengingat ukuran dataset yang relatif terbatas.

### Alur Pemrosesan Awal (*Preprocessing*)
- **Standardisasi Resolusi:** Seluruh citra diubah ukurannya (*resize*) menjadi 160×160 piksel, menyesuaikan dengan format masukan standar arsitektur MobileNetV2.
- **Transmutasi Nilai Piksel:** Skala piksel dinormalisasi ke dalam rentang [-1, 1] melalui fungsi `mobilenet_v2.preprocess_input`, berbeda dengan metode normalisasi absolut (pembagian 255.0).
- **Akselerasi Pipeline Data:** Penggunaan rantai metode `.cache().shuffle().prefetch()` diterapkan guna memastikan aliran pasokan data ke unit pemroses berjalan konstan dan menghindari antrean pemrosesan I/O.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    CATS_DOGS_DIR, validation_split=0.3, subset="training",
    seed=123, image_size=(160, 160), batch_size=32
)
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    CATS_DOGS_DIR, validation_split=0.3, subset="validation",
    seed=123, image_size=(160, 160), batch_size=32
)


class_names_tl = train_ds.class_names
print("Kelas Transfer Learning:", class_names_tl)


val_batches = tf.data.experimental.cardinality(val_test_ds)
test_ds = val_test_ds.take(val_batches // 2)
val_ds  = val_test_ds.skip(val_batches // 2)


preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
val_ds   = val_ds.map(  lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
test_ds  = test_ds.map( lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3), include_top=False, weights=None
)
base_model.trainable = False

tl_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
], name='Transfer_Learning_MobileNetV2')

tl_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

tl_model.summary()
tl_total_params     = tl_model.count_params()
tl_trainable_params = sum([tf.size(w).numpy() for w in tl_model.trainable_weights])
print(f"\nTotal parameter TL     : {tl_total_params:,}")
print(f"Parameter yang dilatih : {tl_trainable_params:,}")

print("\nMemulai pelatihan Transfer Learning...")
start_time = time.time()
tl_history = tl_model.fit(train_ds, epochs=5, validation_data=val_ds, verbose=1)
tl_time_total = time.time() - start_time
print(f"Selesai! Waktu training Transfer Learning: {tl_time_total:.2f} detik")

---
## Evaluasi & Visualisasi Hasil

In [ ]:
import os
os.makedirs('figures', exist_ok=True)


def plot_metrics(history, title, color, save_path):
    acc      = history.history['accuracy']
    val_acc  = history.history['val_accuracy']
    loss     = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc)+1)

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    axes[0].plot(epochs_range, acc,     label='Training',   color=color,  linewidth=2)
    axes[0].plot(epochs_range, val_acc, label='Validation', color='coral',linewidth=2, linestyle='--')
    axes[0].set_title(f'Akurasi - {title}', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylim([0, 1])

    axes[1].plot(epochs_range, loss,     label='Training',   color=color,  linewidth=2)
    axes[1].plot(epochs_range, val_loss, label='Validation', color='coral',linewidth=2, linestyle='--')
    axes[1].set_title(f'Loss - {title}', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_metrics(cnn_history, 'CNN From Scratch', 'steelblue', 'figures/training_curves_cnn.png')
plot_metrics(tl_history,  'Transfer Learning (MobileNetV2)', 'seagreen', 'figures/training_curves_tl.png')


cnn_loss, cnn_acc = cnn_model.evaluate(X_test_cnn, y_test_cnn, verbose=0)
print(f"\n[CNN Scratch] Loss: {cnn_loss:.4f} | Akurasi Testing: {cnn_acc*100:.2f}%")

y_pred_cnn = (cnn_model.predict(X_test_cnn, verbose=0) > 0.5).astype("int32")
print("\nClassification Report - CNN From Scratch:")
print(classification_report(y_test_cnn, y_pred_cnn, target_names=['Airplane','Automobile']))

cm_cnn = confusion_matrix(y_test_cnn, y_pred_cnn)
plt.figure(figsize=(4.2, 3.6))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Airplane','Automobile'],
            yticklabels=['Airplane','Automobile'])
plt.title('Confusion Matrix - CNN from Scratch', fontweight='bold')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('figures/confusion_matrix_cnn.png', dpi=150, bbox_inches='tight')
plt.show()


tl_loss, tl_acc = tl_model.evaluate(test_ds, verbose=0)
print(f"\n[Transfer Learning] Loss: {tl_loss:.4f} | Akurasi Testing: {tl_acc*100:.2f}%")

y_true_tl = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_tl = (tl_model.predict(test_ds, verbose=0) > 0.5).astype("int32").flatten()
print("\nClassification Report - Transfer Learning:")
print(classification_report(y_true_tl, y_pred_tl, target_names=class_names_tl))

cm_tl = confusion_matrix(y_true_tl, y_pred_tl)
plt.figure(figsize=(4.2, 3.6))
sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names_tl, yticklabels=class_names_tl)
plt.title('Confusion Matrix - Transfer Learning (MobileNetV2)', fontweight='bold')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('figures/confusion_matrix_tl.png', dpi=150, bbox_inches='tight')
plt.show()


y_pred_cnn_flat = y_pred_cnn.flatten()
y_true_cnn_flat = y_test_cnn.flatten().astype(int)
names_cnn = ['Airplane', 'Automobile']

correct_idx = np.where(y_pred_cnn_flat == y_true_cnn_flat)[0][:4]
wrong_idx   = np.where(y_pred_cnn_flat != y_true_cnn_flat)[0][:4]

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, idx in enumerate(correct_idx):
    axes[0, i].imshow(X_test_cnn[idx])
    axes[0, i].set_title(f"True:{names_cnn[y_true_cnn_flat[idx]]}\nPred:{names_cnn[y_pred_cnn_flat[idx]]}", fontsize=8, color='green')
    axes[0, i].axis('off')
for i, idx in enumerate(wrong_idx):
    axes[1, i].imshow(X_test_cnn[idx])
    axes[1, i].set_title(f"True:{names_cnn[y_true_cnn_flat[idx]]}\nPred:{names_cnn[y_pred_cnn_flat[idx]]}", fontsize=8, color='red')
    axes[1, i].axis('off')
fig.text(0.02, 0.75, 'Benar', fontsize=11, color='green', fontweight='bold', rotation=90, va='center')
fig.text(0.02, 0.25, 'Salah', fontsize=11, color='red',   fontweight='bold', rotation=90, va='center')
plt.suptitle('Contoh Prediksi - CNN From Scratch', fontweight='bold')
plt.tight_layout(rect=[0.03, 0, 1, 1])
plt.savefig('figures/predictions_cnn.png', dpi=150, bbox_inches='tight')
plt.show()


test_images_tl, test_labels_tl = [], []
for images, labels in test_ds:
    test_images_tl.append(images.numpy())
    test_labels_tl.append(labels.numpy())
test_images_tl = np.concatenate(test_images_tl)
test_labels_tl = np.concatenate(test_labels_tl)


display_images_tl = (test_images_tl + 1.0) / 2.0
display_images_tl = np.clip(display_images_tl, 0, 1)

correct_idx_tl = np.where(y_pred_tl == y_true_tl)[0][:4]
wrong_idx_tl   = np.where(y_pred_tl != y_true_tl)[0][:4]

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, idx in enumerate(correct_idx_tl):
    axes[0, i].imshow(display_images_tl[idx])
    axes[0, i].set_title(f"True:{class_names_tl[y_true_tl[idx]]}\nPred:{class_names_tl[y_pred_tl[idx]]}", fontsize=8, color='green')
    axes[0, i].axis('off')
for i, idx in enumerate(wrong_idx_tl):
    axes[1, i].imshow(display_images_tl[idx])
    axes[1, i].set_title(f"True:{class_names_tl[y_true_tl[idx]]}\nPred:{class_names_tl[y_pred_tl[idx]]}", fontsize=8, color='red')
    axes[1, i].axis('off')
fig.text(0.02, 0.75, 'Benar', fontsize=11, color='green', fontweight='bold', rotation=90, va='center')
fig.text(0.02, 0.25, 'Salah', fontsize=11, color='red',   fontweight='bold', rotation=90, va='center')
plt.suptitle('Contoh Prediksi - Transfer Learning (MobileNetV2)', fontweight='bold')
plt.tight_layout(rect=[0.03, 0, 1, 1])
plt.savefig('figures/predictions_tl.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

cnn_train_acc  = max(cnn_history.history['accuracy'])
cnn_val_acc    = max(cnn_history.history['val_accuracy'])
cnn_train_loss = min(cnn_history.history['loss'])
cnn_val_loss   = min(cnn_history.history['val_loss'])

tl_train_acc  = max(tl_history.history['accuracy'])
tl_val_acc    = max(tl_history.history['val_accuracy'])
tl_train_loss = min(tl_history.history['loss'])
tl_val_loss   = min(tl_history.history['val_loss'])

cnn_gap = cnn_train_acc - cnn_val_acc
tl_gap  = tl_train_acc  - tl_val_acc

risiko_cnn = "Tinggi" if cnn_gap > 0.10 else ("Sedang" if cnn_gap > 0.05 else "Rendah")
risiko_tl  = "Tinggi" if tl_gap  > 0.10 else ("Sedang" if tl_gap  > 0.05 else "Rendah")

comparison_table = [
    ("Akurasi Training",        f"{cnn_train_acc*100:.2f}%",      f"{tl_train_acc*100:.2f}%"),
    ("Akurasi Validation",      f"{cnn_val_acc*100:.2f}%",        f"{tl_val_acc*100:.2f}%"),
    ("Akurasi Testing",         f"{cnn_acc*100:.2f}%",            f"{tl_acc*100:.2f}%"),
    ("Loss Training",           f"{cnn_train_loss:.4f}",          f"{tl_train_loss:.4f}"),
    ("Loss Validation",         f"{cnn_val_loss:.4f}",            f"{tl_val_loss:.4f}"),
    ("Waktu Training",          f"{cnn_time_total:.2f} detik",    f"{tl_time_total:.2f} detik"),
    ("Jumlah Parameter",        f"{cnn_total_params:,}",          f"{tl_total_params:,} (trainable: {tl_trainable_params:,})"),
    ("Risiko Overfitting",      f"{risiko_cnn} (gap {cnn_gap*100:.1f}%)", f"{risiko_tl} (gap {tl_gap*100:.1f}%)"),
    ("Kemudahan Implementasi",  "Sedang - perlu desain arsitektur sendiri", "Mudah - arsitektur sudah tersedia"),
    ("Kesesuaian dengan Dataset", "Cukup - dataset kecil membatasi performa", "Baik - fitur ImageNet relevan untuk objek umum"),
]

print("="*78)
print(f"{'Aspek':<28}{'CNN from Scratch':<26}{'Transfer Learning':<24}")
print("="*78)
for aspek, cnn_val, tl_val in comparison_table:
    print(f"{aspek:<28}{cnn_val:<26}{tl_val:<24}")
print("="*78)


categories  = ['Akurasi\nTraining', 'Akurasi\nValidation', 'Akurasi\nTesting']
cnn_scores  = [cnn_train_acc, cnn_val_acc, cnn_acc]
tl_scores   = [tl_train_acc,  tl_val_acc,  tl_acc]

x = np.arange(len(categories))
width = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
bars1 = ax.bar(x - width/2, [s*100 for s in cnn_scores], width, label='CNN from Scratch', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, [s*100 for s in tl_scores],  width, label='Transfer Learning',color='seagreen',  alpha=0.85)
ax.set_ylabel('Akurasi (%)'); ax.set_title('Perbandingan Akurasi', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(categories)
ax.set_ylim([0, 110]); ax.legend(); ax.grid(axis='y', alpha=0.3)
for bar in bars1: ax.annotate(f'{bar.get_height():.1f}%', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()), xytext=(0,3), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2: ax.annotate(f'{bar.get_height():.1f}%', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()), xytext=(0,3), textcoords='offset points', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('figures/comparison_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Analisis

### 1. Analisis Komprehensif Dataset

* **Kapasitas Dataset untuk CNN Independen:** Penggunaan 700 sampel *training* dari CIFAR-10 sangat sub-optimal untuk melatih CNN dari awal (*from scratch*). Secara empiris, arsitektur CNN konvensional membutuhkan setidaknya 5.000–10.000 citra per kelas agar fitur yang diekstraksi bersifat *robust*. Keterbatasan kuantitas ini secara drastis memicu terjadinya *overfitting*.
* **Resolusi dan Variasi Visual:** Meskipun resolusi spasial CIFAR-10 sangat rendah (32×32 piksel) yang membatasi informasi tekstur granular, dataset ini memiliki variasi sudut pandang, pencahayaan, dan latar belakang yang memadai untuk menunjang generalisasi model dasar.
* **Keseimbangan Distribusi Kelas:** Kelas *Airplane* dan *Automobile* terdistribusi secara proporsional (rasio ~50:50). Hal ini mengeliminasi risiko bias kelas yang sering terjadi pada klasifikasi biner, sehingga metrik akurasi dapat dijadikan parameter evaluasi yang valid.
* **Kompleksitas Derau (*Noise*) dan Fitur:** Resolusi rendah pada CIFAR-10 menghasilkan tingkat presisi detail yang terbatas. Sebagai pembanding, dataset *Cats vs Dogs* menghadirkan kompleksitas tingkat lanjut melalui variasi pose dan rentang latar belakang yang dinamis.
* **Korelasi Kualitas Data dan Arsitektur:** Performa CNN *from scratch* memiliki korelasi absolut dengan kuantitas data. Pada 700 sampel, arsitektur akan kesulitan menemukan pola fundamental. Sebaliknya, metode *Transfer Learning* jauh lebih resilien terhadap dataset minor karena memanfaatkan perpustakaan fitur prabangun yang sangat kaya.

---

### 2. Evaluasi Kinerja Komparatif Model

* **Superioritas Performa Metrik:** Pendekatan *Transfer Learning* (MobileNetV2) secara konsisten menghasilkan akurasi *testing* yang jauh lebih superior dalam siklus konvergensi yang lebih pendek (5 *epoch* vs 10 *epoch*). Hal ini memvalidasi keunggulan fitur *pretrained* ketika dihadapkan pada kelangkaan data.
* **Validitas Analisis Metrik:** Menilai model semata-mata dari akurasi global dapat menghasilkan konklusi yang bias. Penggunaan *Confusion Matrix* dan evaluasi presisi lintas-metrik (*Precision, Recall, F1-Score*) pada *Classification Report* memberikan wawasan mendalam terkait kapabilitas model dalam mendiskriminasi kelas.
* **Indikasi dan Mitigasi Overfitting:** Arsitektur CNN *from scratch* menunjukkan *gap* marjinal yang lebih lebar antara kurva *training* dan *validation*, yang merupakan manifestasi kuat dari *overfitting*.
* **Stabilitas Fase Pelatihan:** Pembekuan parameter (*layer freezing*) pada MobileNetV2 mengeliminasi fluktuasi pembaruan bobot, menghasilkan lintasan akurasi validasi yang stabil. Sebaliknya, pembaruan seluruh parameter *trainable* pada setiap *epoch* dalam model CNN mandiri menyebabkan ketidakstabilan *loss*.

---

### 3. Matriks Pemilihan Pendekatan

Untuk menyederhanakan proses pengambilan keputusan dalam merancang arsitektur klasifikasi citra, panduan matriks berikut dapat digunakan:

| Parameter / Kondisi | Rekomendasi: CNN *From Scratch* | Rekomendasi: *Transfer Learning* |
| :--- | :--- | :--- |
| **Volume Dataset** | Masif (>100.000 citra per kelas) | Skala minor hingga menengah (<50.000 citra) |
| **Karakteristik Domain** | Spesifik / Anomali visual tinggi (mis. citra radiologi, termal) | Berkorelasi dengan citra umum di *ImageNet* |
| **Sumber Daya Komputasi**| Melimpah (Cluster GPU/TPU) dengan alokasi waktu tak terbatas | Terbatas (CPU/GPU *entry-level*), komputasi instan |
| **Fokus Pengembangan** | Kontrol arsitektur penuh, optimasi ukuran perangkat *edge* | Pembuatan purwarupa (*prototyping*) presisi tinggi |

---

### 4. Studi Kasus dan Pengambilan Keputusan

* **Skenario 1: Diagnostik Medis dengan Data Terbatas (300 citra)**
    * **Keputusan:** *Transfer Learning (Feature Extraction)*
    * **Justifikasi:** Menggunakan CNN *from scratch* dipastikan akan menghasilkan *overfitting* fatal. Mengekstraksi fitur menggunakan model *pretrained* yang dibekukan memastikan model dilatih berbasis fondasi pengenalan pola hierarkis. *Fine-tuning* parsial pada beberapa *layer* akhir dapat dilakukan secara hati-hati untuk spesialisasi visual medis.

* **Skenario 2: Repositori Produk Internal Korporat (1 Juta citra)**
    * **Keputusan:** *Transfer Learning + Full Fine-Tuning* (Pendekatan Hibrida)
    * **Justifikasi:** Meskipun volume dataset sangat memadai untuk CNN *from scratch*, menginisiasi pelatihan dasar dari bobot *pretrained* terbukti jauh lebih efisien untuk memangkas waktu *training*.

* **Skenario 3: Eksekusi Purwarupa Cepat (500 citra, Tenggat 2 Hari)**
    * **Keputusan:** *Transfer Learning (Feature Extraction)*
    * **Justifikasi:** Limitasi waktu mengeliminasi kemungkinan riset arsitektur maupun siklus *hyperparameter tuning* yang panjang. MobileNetV2 dengan pengklasifikasi sederhana dapat mencapai metrik *baseline* yang optimal hanya dalam hitungan menit komputasi.

* **Skenario 4: Domain Ekstrem Spesifik dengan Infrastruktur GPU Masif**
    * **Keputusan:** *Transfer Learning + Full Fine-Tuning*
    * **Justifikasi:** Strategi perintis *State-of-The-Art* (SOTA) kontemporer membuktikan bahwa inisialisasi bobot *transfer learning* memfasilitasi percepatan konvergensi 3 hingga 10 kali lipat lebih optimal dibandingkan inisialisasi acak (*random initialization*), tak memandang betapa uniknya domain dataset.

---
## Kesimpulan

Eksperimen ini membandingkan dua pendekatan klasifikasi citra secara langsung:

| Kesimpulan | CNN from Scratch | Transfer Learning |
|---|---|---|
| Cocok untuk | Dataset besar, domain unik | Dataset kecil, domain umum |
| Kecepatan training | Lebih lambat | Lebih cepat |
| Akurasi (dataset kecil) | Lebih rendah | Lebih tinggi |
| Risiko overfitting | Lebih tinggi | Lebih rendah |
| Fleksibilitas arsitektur | Penuh | Terbatas pada base model |

**Rekomendasi:** Untuk mayoritas kasus praktis di industri, Transfer Learning adalah titik awal yang lebih baik. CNN from scratch baru dipertimbangkan ketika dataset sangat besar dan domain benar-benar unik.

---

## Refleksi Pribadi

**1. Tantangan Terbesar**  
Tantangan utama yang saya hadapi adalah merancang pipeline data yang efisien menggunakan `tf.data`, terutama dalam menerapkan standar *preprocessing* yang spesifik untuk model Transfer Learning. Sebagai contoh, arsitektur MobileNetV2 mensyaratkan normalisasi piksel pada rentang [-1, 1], berbeda dengan CNN konvensional yang umumnya menggunakan rentang [0, 1]. Ketidaksesuaian pada tahap ini sering kali tidak memunculkan pesan *error* secara eksplisit, namun berakibat fatal pada penurunan akurasi model.

**2. Aspek Paling Kompleks**  
Secara konseptual, Transfer Learning memiliki tingkat kerumitan tersendiri. Pendekatan ini menuntut pemahaman komprehensif mengenai arsitektur *pretrained model*, strategi *freezing layer* yang tepat sasaran, serta penyesuaian dimensi input dan *preprocessing* agar selaras dengan karakteristik model asalnya.

**3. Perbandingan Pengalaman**  
Membangun CNN dari awal (*from scratch*) memberikan pemahaman fundamental karena setiap lapisan arsitektur harus dirancang secara manual. Sebaliknya, Transfer Learning ibarat "berdiri di atas bahu raksasa"—kita dapat mendaur ulang fitur visual kompleks yang telah dipelajari model dari jutaan citra, sehingga proses *training* menjadi jauh lebih singkat dan tingkat konvergensi lebih cepat tercapai.

**4. Preferensi untuk Implementasi di Dunia Nyata**  
Untuk kasus dunia nyata yang umumnya melibatkan dataset berskala kecil hingga menengah, saya akan memprioritaskan Transfer Learning. Pendekatan ini menawarkan reliabilitas yang lebih tinggi, menghemat beban komputasi, serta secara signifikan mempercepat siklus pengembangan hingga fase *deployment*.

**5. Wawasan Baru yang Diperoleh**  
Pelajaran paling berharga dari eksperimen ini adalah bahwa merancang sistem *deep learning* tidak hanya sekadar mengejar metrik akurasi tertinggi. Dibutuhkan kemampuan analisis *trade-off* secara holistik antara ketersediaan data, batas waktu, kapasitas komputasi, dan mitigasi risiko *overfitting*. Tidak ada satu metode yang superior untuk segala kondisi; konteks dan batasan masalah adalah penentu utama.